# Sanskrit LLM Fine-Tuning — End-to-End Notebook

Single-notebook version of the project (data prep → tokenizer inspection →
QLoRA fine-tuning → inference → evaluation). Originally split across
`scripts/prepare_data.py`, `scripts/inspect_tokenizer.py`, `scripts/train.py`,
`scripts/infer.py`, and `eval/evaluate.py`; consolidated here so the whole
pipeline can run top-to-bottom in one Colab session.

**Run order:** Setup → Config → Data Prep → Tokenizer Inspection →
Training → Inference → Evaluation. Each section is self-contained but
depends on artifacts (files/variables) produced by the previous one.


## 1. Setup — install dependencies

In [9]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets sacrebleu pandas


In [10]:
import json
import os
import random
import argparse
from pathlib import Path
from collections import defaultdict

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import sacrebleu

random.seed(42)


## 2. Config

All the knobs that used to be CLI flags across the different scripts now
live here as a single config dict, so you only need to edit one place.


In [11]:
from huggingface_hub import login
login()

In [12]:
CONFIG = {
    # --- data prep ---
    "max_pairs": 6000,
    "val_frac": 0.05,
    "test_frac": 0.05,
    "data_dir": "data",
    "itihasa_local_path": None,  # set to a local parquet/json/jsonl path if HF auto-download fails

    # --- model / training ---
    "base_model": "meta-llama/Llama-3.2-1B-Instruct",  # final pick; swap to "Qwen/Qwen2.5-1.5B-Instruct" for the ungated alternative
    "output_dir": "outputs/lora-sanskrit",
    "epochs": 2.0,
    "lr": 2e-4,
    "per_device_batch_size": 4,
    "grad_accum": 4,
    "max_seq_len": 512,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "max_train_samples": 500,  # cap for a quick smoke-test run; set None for full run

    # --- inference / eval ---
    "max_new_tokens": 200,
    "eval_limit": 100,  # cap number of test examples run through inference, None = all
}

Path(CONFIG["data_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path("eval").mkdir(parents=True, exist_ok=True)
CONFIG


{'max_pairs': 6000,
 'val_frac': 0.05,
 'test_frac': 0.05,
 'data_dir': 'data',
 'itihasa_local_path': None,
 'base_model': 'meta-llama/Llama-3.2-1B-Instruct',
 'output_dir': 'outputs/lora-sanskrit',
 'epochs': 2.0,
 'lr': 0.0002,
 'per_device_batch_size': 4,
 'grad_accum': 4,
 'max_seq_len': 512,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'max_train_samples': 500,
 'max_new_tokens': 200,
 'eval_limit': 100}

## 3. Data Preparation

Builds an instruction-tuning dataset for Sanskrit↔English tasks from the
`itihasa` parallel corpus (with backup sources and a curated seed set as
fallbacks), then expands each parallel pair into multiple instruction task
types (translation both directions, explanation, QA, summarization) so the
model learns instruction-following, not just one task shape.

See `REPORT.md` section 2 for full reasoning on these design choices.


In [13]:
SA2EN_TEMPLATES = [
    "Translate the following Sanskrit text into English.",
    "Provide an English translation of this Sanskrit sentence.",
    "What does this Sanskrit passage mean in English?",
]

EN2SA_TEMPLATES = [
    "Translate the following English text into Sanskrit.",
    "Provide a Sanskrit translation of this English sentence.",
    "Render this English sentence in Sanskrit.",
]

EXPLAIN_TEMPLATES = [
    "Explain the meaning and context of this Sanskrit verse in simple English.",
    "Give a brief explanation of what this Sanskrit passage is conveying.",
]

QA_TEMPLATES = [
    "Based on the Sanskrit verse below, answer the question in English.\nQuestion: What is the verse saying, in one sentence?",
    "Read the Sanskrit verse and answer: What is the main idea expressed here?",
]

SUMMARY_TEMPLATES = [
    "Summarize the following Sanskrit passage in one or two English sentences.",
]


def build_examples_from_pair(sa: str, en: str, idx: int):
    """Turn one (Sanskrit, English) pair into several instruction examples."""
    examples = []

    examples.append({
        "instruction": random.choice(SA2EN_TEMPLATES),
        "input": sa,
        "output": en,
        "task_type": "sa2en",
    })

    examples.append({
        "instruction": random.choice(EN2SA_TEMPLATES),
        "input": en,
        "output": sa,
        "task_type": "en2sa",
    })

    # Explanation task re-uses the English side as the "explanation" target.
    # Simplifying assumption (documented in REPORT.md): true explanations
    # would need a separate commentary corpus, which is scarce/inconsistent
    # in license and quality.
    if idx % 3 == 0:
        examples.append({
            "instruction": random.choice(EXPLAIN_TEMPLATES),
            "input": sa,
            "output": f"This verse conveys: {en}",
            "task_type": "explain",
        })

    if idx % 4 == 0:
        examples.append({
            "instruction": random.choice(QA_TEMPLATES),
            "input": sa,
            "output": en,
            "task_type": "qa",
        })

    if idx % 5 == 0 and len(en.split()) > 6:
        examples.append({
            "instruction": random.choice(SUMMARY_TEMPLATES),
            "input": sa,
            "output": en,
            "task_type": "summary",
        })

    return examples


In [14]:
def _extract_pairs_from_rows(rows, max_pairs, pairs):
    for row in rows:
        # itihasa schema: {"translation": {"sn": "...", "en": "..."}}
        tr = row.get("translation", row)
        sa = (tr.get("sn") or tr.get("sa") or "").strip()
        en = (tr.get("en") or "").strip()
        if sa and en and len(sa) > 3 and len(en) > 3:
            pairs.append((sa, en))
        if len(pairs) >= max_pairs:
            break
    return pairs


def load_itihasa(max_pairs: int):
    """Load the itihasa Sanskrit-English parallel corpus from Hugging Face.

    NOTE: HF deprecated dataset *loading scripts* (the old itihasa.py path).
    We work around this by loading the dataset's auto-generated parquet
    files directly via the `refs/convert/parquet` branch HF maintains for
    every dataset. Falls back to an empty list so the pipeline still runs
    end-to-end on the bundled seed set alone rather than crashing.
    """
    pairs = []

    try:
        print("Loading rahular/itihasa via parquet (refs/convert/parquet) ...")
        ds = load_dataset(
            "parquet",
            data_files={
                "train": "hf://datasets/rahular/itihasa@~parquet/default/train/0000.parquet",
                "validation": "hf://datasets/rahular/itihasa@~parquet/default/validation/0000.parquet",
                "test": "hf://datasets/rahular/itihasa@~parquet/default/test/0000.parquet",
            },
        )
        for split in ds:
            pairs = _extract_pairs_from_rows(ds[split], max_pairs, pairs)
            if len(pairs) >= max_pairs:
                break
        if pairs:
            print(f"Loaded {len(pairs)} pairs from itihasa (parquet).")
            return pairs
    except Exception as e:
        print(f"Parquet load attempt failed ({e}); trying revision='refs/convert/parquet' ...")

    try:
        ds = load_dataset("rahular/itihasa", revision="refs/convert/parquet")
        for split in ds:
            pairs = _extract_pairs_from_rows(ds[split], max_pairs, pairs)
            if len(pairs) >= max_pairs:
                break
        if pairs:
            print(f"Loaded {len(pairs)} pairs from itihasa (revision fallback).")
            return pairs
    except Exception as e:
        print(f"revision='refs/convert/parquet' attempt failed ({e}).")

    print("Could not load itihasa from any known path; continuing with the "
          "bundled seed set only. See REPORT.md for how this was handled.")
    return pairs


def load_backup_corpus(max_pairs: int):
    """Second, independent Sanskrit-English source, tried only if itihasa
    yields too few pairs to be useful for fine-tuning."""
    pairs = []
    candidates = [
        "rahular/itihasa",
        "Sanskrit-nlp/Sanskrit_English_Parallel_Corpus",
        "ganga4364/Sanskrit-English-Sentence-Pairs",
    ]

    for name in candidates:
        try:
            print(f"Trying backup source: {name} ...")
            ds = load_dataset(name)
            for split in ds:
                for row in ds[split]:
                    sa = (row.get("sanskrit") or row.get("sn") or row.get("sa")
                          or (row.get("translation", {}) or {}).get("sn", "")).strip()
                    en = (row.get("english") or row.get("en")
                          or (row.get("translation", {}) or {}).get("en", "")).strip()
                    if sa and en and len(sa) > 3 and len(en) > 3:
                        pairs.append((sa, en))
                    if len(pairs) >= max_pairs:
                        break
                if len(pairs) >= max_pairs:
                    break
            if pairs:
                print(f"Loaded {len(pairs)} pairs from backup source {name}.")
                return pairs
        except Exception as e:
            print(f"  backup source {name} failed: {e}")
            continue

    return pairs


def load_itihasa_from_file(path: str, max_pairs: int):
    """Manual fallback: load itihasa (or similarly-shaped parallel data)
    from a local file, in case automatic `datasets` loading keeps failing.
    Supports .parquet, .json, and .jsonl.
    """
    import pandas as pd

    pairs = []
    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    elif path.endswith(".jsonl"):
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_json(path)

    for _, row in df.iterrows():
        row = row.to_dict()
        tr = row.get("translation", row)
        if isinstance(tr, str):
            continue
        sa = (tr.get("sn") or tr.get("sa") or "").strip() if tr else ""
        en = (tr.get("en") or "").strip() if tr else ""
        if sa and en and len(sa) > 3 and len(en) > 3:
            pairs.append((sa, en))
        if len(pairs) >= max_pairs:
            break

    print(f"Loaded {len(pairs)} pairs from local file {path}.")
    return pairs


def load_gita_fallback():
    """Small bundled seed set (Bhagavad Gita opening verses + well-known
    subhashitas), also used as a curated, hand-verifiable slice of the
    test set."""
    seed = [
        ("धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः। मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय॥",
         "On the field of dharma, the field of Kuru, assembled and eager to fight, what did my sons and the sons of Pandu do, O Sanjaya?"),
        ("कर्मण्येवाधिकारस्ते मा फलेषु कदाचन। मा कर्मफलहेतुर्भूर्मा ते सङ्गोऽस्त्वकर्मणि॥",
         "You have a right to perform your prescribed duties, but you are not entitled to the fruits of your actions. Never consider yourself the cause of the results, and never be attached to inaction."),
        ("योगस्थः कुरु कर्माणि सङ्गं त्यक्त्वा धनञ्जय। सिद्ध्यसिद्ध्योः समो भूत्वा समत्वं योग उच्यते॥",
         "Perform your duty established in yoga, O Dhananjaya, abandoning attachment, and be even-minded in success and failure. Such evenness of mind is called yoga."),
        ("विद्या ददाति विनयं विनयाद्याति पात्रताम्। पात्रत्वाद्धनमाप्नोति धनाद्धर्मं ततः सुखम्॥",
         "Knowledge gives discipline, from discipline comes worthiness, from worthiness one attains wealth, from wealth comes righteous conduct, and from that, happiness."),
        ("अहिंसा परमो धर्मः धर्म हिंसा तथैव च। धर्मे व्यवस्थितो हिंसा तस्मादहिंसा परमो धर्मः॥",
         "Non-violence is the highest duty; yet non-violence used for the protection of duty is also duty. Duty enacted in righteousness is non-violence; hence non-violence is the highest duty."),
        ("सत्यमेव जयते नानृतं सत्येन पन्था विततो देवयानः।",
         "Truth alone triumphs, not falsehood. Through truth the divine path is spread out."),
        ("वसुधैव कुटुम्बकम्।",
         "The whole world is one family."),
        ("न त्वेवाहं जातु नासं न त्वं नेमे जनाधिपाः। न चैव न भविष्यामः सर्वे वयमतः परम्॥",
         "Never was there a time when I did not exist, nor you, nor all these kings; nor in the future shall any of us cease to be."),
        ("उद्यमेन हि सिध्यन्ति कार्याणि न मनोरथैः। न हि सुप्तस्य सिंहस्य प्रविशन्ति मुखे मृगाः॥",
         "Tasks are accomplished through effort, not through mere wishes. Deer do not walk into the mouth of a sleeping lion."),
        ("यत्र नार्यस्तु पूज्यन्ते रमन्ते तत्र देवताः।",
         "Where women are honored, there the divine beings rejoice."),
    ]
    return seed


In [15]:
def prepare_data(config):
    out_dir = Path(config["data_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    pairs = []

    if config.get("itihasa_local_path"):
        try:
            pairs = load_itihasa_from_file(config["itihasa_local_path"], config["max_pairs"])
        except Exception as e:
            print(f"Could not load local file {config['itihasa_local_path']} ({e}); "
                  f"trying automatic download instead.")

    if not pairs:
        try:
            pairs = load_itihasa(config["max_pairs"])
        except Exception as e:
            print(f"Could not load itihasa ({e}); trying backup sources.")
            pairs = []

    MIN_USABLE_PAIRS = 200
    if len(pairs) < MIN_USABLE_PAIRS:
        print(f"Only {len(pairs)} pairs from itihasa (need >= {MIN_USABLE_PAIRS}); "
              f"trying backup corpora ...")
        try:
            backup_pairs = load_backup_corpus(config["max_pairs"])
            pairs.extend(backup_pairs)
        except Exception as e:
            print(f"Backup corpus loading also failed: {e}")

    if len(pairs) < MIN_USABLE_PAIRS:
        print(
            f"\n*** WARNING: only {len(pairs)} real parallel pairs available "
            f"(plus the 10-verse curated seed set). This is too little data "
            f"for meaningful fine-tuning. ***\n"
            "Fix options, in order of preference:\n"
            "  1. In Colab, restart runtime and re-run this cell (transient HF/network issues are common).\n"
            "  2. pip install -U datasets\n"
            "  3. Manually download from https://huggingface.co/datasets/rahular/itihasa/tree/main "
            "and set CONFIG['itihasa_local_path'].\n"
            "  4. Proceed with the small seed set for a PIPELINE SMOKE TEST only.\n"
        )

    seed_pairs = load_gita_fallback()

    all_pairs = pairs
    random.shuffle(all_pairs)

    all_examples = []
    for i, (sa, en) in enumerate(all_pairs):
        all_examples.extend(build_examples_from_pair(sa, en, i))

    random.shuffle(all_examples)

    n = len(all_examples)
    n_val = int(n * config["val_frac"])
    n_test = int(n * config["test_frac"])

    val = all_examples[:n_val]
    test = all_examples[n_val:n_val + n_test]
    train = all_examples[n_val + n_test:]

    seed_examples = []
    for i, (sa, en) in enumerate(seed_pairs):
        seed_examples.extend(build_examples_from_pair(sa, en, i))
    test.extend(seed_examples)

    def dump(rows, path):
        with open(path, "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    dump(train, out_dir / "train.jsonl")
    dump(val, out_dir / "val.jsonl")
    dump(test, out_dir / "test.jsonl")

    print(f"train={len(train)}  val={len(val)}  test={len(test)}")
    print(f"Written to {out_dir}/")


prepare_data(CONFIG)


Loading rahular/itihasa via parquet (refs/convert/parquet) ...
Parquet load attempt failed (Unable to find 'hf://datasets/rahular/itihasa@~parquet/default/train/0000.parquet'); trying revision='refs/convert/parquet' ...
Loaded 6000 pairs from itihasa (revision fallback).
train=15030  val=835  test=864
Written to data/


## 4. Tokenizer Inspection

Quick diagnostic: how badly does the base model's tokenizer fragment
Sanskrit (Devanagari) text compared to English text of similar length?
Directly addresses the "Tokenization / Vocabulary" challenge in the
assignment brief, and is useful evidence when justifying a base-model
choice in the report.


In [16]:
TOKENIZER_SAMPLES = [
    ("वसुधैव कुटुम्बकम्।", "The whole world is one family."),
    ("धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः।",
     "On the field of dharma, the field of Kuru, assembled and eager to fight."),
    ("कर्मण्येवाधिकारस्ते मा फलेषु कदाचन।",
     "You have a right to perform your duty, but not to the fruits of action."),
    ("सत्यमेव जयते।", "Truth alone triumphs."),
    ("अहिंसा परमो धर्मः।", "Non-violence is the highest duty."),
]


def inspect_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)

    print(f"Tokenizer: {model_name}")
    print(f"Vocab size: {tok.vocab_size}\n")

    report = []
    for sa, en in TOKENIZER_SAMPLES:
        sa_ids = tok.encode(sa, add_special_tokens=False)
        en_ids = tok.encode(en, add_special_tokens=False)

        sa_tpc = len(sa_ids) / max(len(sa), 1)
        en_tpc = len(en_ids) / max(len(en), 1)

        print(f"SA: {sa}  chars={len(sa)} tokens={len(sa_ids)} tokens/char={sa_tpc:.2f}")
        print(f"EN: {en}  chars={len(en)} tokens={len(en_ids)} tokens/char={en_tpc:.2f}")
        print(f"  --> Sanskrit costs {sa_tpc / en_tpc:.2f}x more tokens/char than English\n")

        report.append({
            "sanskrit": sa, "english": en,
            "sa_chars": len(sa), "sa_tokens": len(sa_ids), "sa_tokens_per_char": round(sa_tpc, 3),
            "en_chars": len(en), "en_tokens": len(en_ids), "en_tokens_per_char": round(en_tpc, 3),
            "ratio_sa_over_en": round(sa_tpc / en_tpc, 2),
        })

    avg_ratio = sum(r["ratio_sa_over_en"] for r in report) / len(report)
    print(f"Average Sanskrit/English tokens-per-char ratio: {avg_ratio:.2f}x")
    return {"model": model_name, "samples": report, "avg_ratio": avg_ratio}


# Compare the candidate base model(s). Add/remove entries as needed.
tokenizer_reports = {}
for m in [CONFIG["base_model"]]:
    tokenizer_reports[m] = inspect_tokenizer(m)

with open("eval/tokenizer_report.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer_reports, f, ensure_ascii=False, indent=2)


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer: meta-llama/Llama-3.2-1B-Instruct
Vocab size: 128000

SA: वसुधैव कुटुम्बकम्।  chars=18 tokens=13 tokens/char=0.72
EN: The whole world is one family.  chars=30 tokens=7 tokens/char=0.23
  --> Sanskrit costs 3.10x more tokens/char than English

SA: धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः।  chars=43 tokens=25 tokens/char=0.58
EN: On the field of dharma, the field of Kuru, assembled and eager to fight.  chars=72 tokens=19 tokens/char=0.26
  --> Sanskrit costs 2.20x more tokens/char than English

SA: कर्मण्येवाधिकारस्ते मा फलेषु कदाचन।  chars=35 tokens=21 tokens/char=0.60
EN: You have a right to perform your duty, but not to the fruits of action.  chars=71 tokens=17 tokens/char=0.24
  --> Sanskrit costs 2.51x more tokens/char than English

SA: सत्यमेव जयते।  chars=13 tokens=6 tokens/char=0.46
EN: Truth alone triumphs.  chars=21 tokens=5 tokens/char=0.24
  --> Sanskrit costs 1.94x more tokens/char than English

SA: अहिंसा परमो धर्मः।  chars=18 tokens=11 tokens/char=0.61
EN: Non-

## 5. Training (QLoRA fine-tuning)

4-bit QLoRA fine-tuning of the base model on the Sanskrit↔English
instruction dataset built above. Designed to run on a single free-tier
Colab GPU (T4 16GB or L4).

**Model choice:** default is `meta-llama/Llama-3.2-1B-Instruct`. Early on
this looked like the wrong call — it trained ~18x slower than
`Qwen/Qwen2.5-1.5B-Instruct` on what I thought was an identical config
(~6 hrs vs ~20 min for 500 samples). Once I added the device-map/attention
diagnostics right after model load (see the print statements below), that
turned out to be an environment-specific bottleneck rather than anything
inherent to the model — with it resolved, Llama trains in ~6 minutes for
500 samples versus Qwen's ~22 minutes on the same config, and it also
fragments Sanskrit less (2.45x tokens/char vs Qwen's 4.02x). Both facts now
favor Llama, which is why it's the default here.
`Qwen/Qwen2.5-1.5B-Instruct` is still wired in as an ungated fallback if you
don't want to deal with Llama's license-acceptance step.


In [17]:
print(f"Loading tokenizer/model: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
)

# Diagnostics: confirm no CPU offload and check attention backend, since both
# can cause dramatic (10-20x) slowdowns on memory-constrained GPUs like a T4.
# `hf_device_map` only exists when accelerate actually had to split the model
# across devices; if it's absent, the whole model fit on one device (good).
device_map = getattr(model, "hf_device_map", None)
if device_map is not None:
    print("Device map:", device_map)
    if any(str(d) == "cpu" for d in device_map.values()):
        print("!! WARNING: some layers offloaded to CPU -- this will be very slow.")
else:
    print("Device map: not set by accelerate -> model loaded fully on a single device:",
          next(model.parameters()).device)
print("Attention implementation:", model.config._attn_implementation)

model = prepare_model_for_kbit_training(model)


Loading tokenizer/model: meta-llama/Llama-3.2-1B-Instruct


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device map: not set by accelerate -> model loaded fully on a single device: cuda:0
Attention implementation: sdpa


In [18]:
def format_example(example, tokenizer):
    """Build a single training string using the model's chat template if it
    has one (instruct models), else fall back to a raw Alpaca-style template."""
    instruction = example["instruction"]
    inp = example.get("input", "")
    output = example["output"]

    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "user", "content": f"{instruction}\n\n{inp}".strip()},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{inp}\n\n"
            f"### Response:\n{output}"
        )
    return {"text": text}


In [19]:
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)

print("Loading datasets ...")
train_ds = load_dataset("json", data_files=f"{CONFIG['data_dir']}/train.jsonl", split="train")
val_ds = load_dataset("json", data_files=f"{CONFIG['data_dir']}/val.jsonl", split="train")

if CONFIG["max_train_samples"]:
    train_ds = train_ds.select(range(min(CONFIG["max_train_samples"], len(train_ds))))

train_ds = train_ds.map(lambda ex: format_example(ex, tokenizer))
val_ds = val_ds.map(lambda ex: format_example(ex, tokenizer))

sft_config = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["per_device_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["lr"],
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    max_length=CONFIG["max_seq_len"],
    dataset_text_field="text",
    report_to="none",
    warmup_steps=56,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
)


Loading datasets ...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/835 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/835 [00:00<?, ? examples/s]

In [20]:
print("Starting training ...")
trainer.train()

print(f"Saving adapter to {CONFIG['output_dir']}")
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

with open(os.path.join(CONFIG["output_dir"], "run_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("Done training.")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Starting training ...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
64,2.401313,2.317279,2.346204,151606.000000,0.542399


Saving adapter to outputs/lora-sanskrit
Done training.


## 6. Inference

Load the base model with (optionally) the fine-tuned LoRA adapter and run
inference — either a single ad-hoc prompt, or batch inference over the test
set (once for the base model to get "before" outputs, once with the adapter
for "after" outputs, so `evaluate.py`'s before/after table has both).


In [21]:
def load_inference_model(base_model, adapter_path=None):
    tok = AutoTokenizer.from_pretrained(adapter_path if adapter_path else base_model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    m = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map="auto",
    )

    if adapter_path:
        m = PeftModel.from_pretrained(m, adapter_path)

    m.eval()
    return m, tok


def generate(m, tok, instruction, input_text, max_new_tokens=200):
    if getattr(tok, "chat_template", None):
        messages = [{"role": "user", "content": f"{instruction}\n\n{input_text}".strip()}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tok.pad_token_id,
        )
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()


def run_batch_inference(base_model, adapter_path, test_file, out_file, max_new_tokens=200, limit=None):
    m, tok = load_inference_model(base_model, adapter_path)

    with open(test_file, encoding="utf-8") as f:
        lines = f.readlines()
    if limit:
        lines = lines[:limit]

    results = []
    for i, line in enumerate(lines):
        ex = json.loads(line)
        pred = generate(m, tok, ex["instruction"], ex.get("input", ""), max_new_tokens)
        results.append({
            "instruction": ex["instruction"],
            "input": ex.get("input", ""),
            "reference": ex["output"],
            "prediction": pred,
            "task_type": ex.get("task_type", "unknown"),
        })
        if (i + 1) % 10 == 0:
            print(f"{i + 1}/{len(lines)} done")

    with open(out_file, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Wrote {len(results)} predictions to {out_file}")

    # free GPU memory before the next model load
    del m
    torch.cuda.empty_cache()
    return results


In [22]:
# "Before" — base model only, no adapter
# CONFIG["eval_limit"]=100
run_batch_inference(
    base_model=CONFIG["base_model"],
    adapter_path=None,
    test_file=f"{CONFIG['data_dir']}/test.jsonl",
    out_file="eval/predictions_base.jsonl",
    max_new_tokens=CONFIG["max_new_tokens"],
    limit=CONFIG["eval_limit"],
)



Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done
Wrote 100 predictions to eval/predictions_base.jsonl


[{'instruction': 'Provide a Sanskrit translation of this English sentence.',
  'input': 'O Śatrughna, arise! Why sleepest you? Bring you at once that lord of the Nisādhas, Guha, Good betide you! he will take the army (over the stream.)',
  'reference': 'शत्रुघ्नोत्तिष्ठ किं शेषे निषादाधिपतिं गुहम्। शीघ्रमानय भद्रं ते तारयिष्यति वाहिनीम्॥',
  'prediction': 'Here\'s the Sanskrit translation of the given English sentence:\n\nओ शत्रugh्ना (O Śatrughna) - "Oh, Śatrughna" (a salutation)\nअरिष्ट (arishtha) - "arise"\nअरिष्ट (arishtha) - "arise"\nअतः (atya) - "at once"\nअतः (atya) - "at once"\nअरिष्ट (arishtha) - "arise"\nअतः (atya) - "at once"\nअरिष्ट (arishtha) - "arise"\nअरिष्ट (arishtha) - "arise"\nअरिष्ट (arishtha) - "arise"\nअरिष्ट (arishtha) - "arise"\nअरिष्ट (arishtha) - "arise"\nअरिष्ट (arishtha) - "ar',
  'task_type': 'en2sa'},
 {'instruction': 'Translate the following English text into Sanskrit.',
  'input': 'And presenting him with many a charming fountain, trees will delight Rāma 

In [23]:
# "After" — base model + fine-tuned LoRA adapter
# CONFIG["eval_limit"] =100
run_batch_inference(
    base_model=CONFIG["base_model"],
    adapter_path=CONFIG["output_dir"],
    test_file=f"{CONFIG['data_dir']}/test.jsonl",
    out_file="eval/predictions.jsonl",
    max_new_tokens=CONFIG["max_new_tokens"],
    limit=CONFIG["eval_limit"],
)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done
Wrote 100 predictions to eval/predictions.jsonl


[{'instruction': 'Provide a Sanskrit translation of this English sentence.',
  'input': 'O Śatrughna, arise! Why sleepest you? Bring you at once that lord of the Nisādhas, Guha, Good betide you! he will take the army (over the stream.)',
  'reference': 'शत्रुघ्नोत्तिष्ठ किं शेषे निषादाधिपतिं गुहम्। शीघ्रमानय भद्रं ते तारयिष्यति वाहिनीम्॥',
  'prediction': 'स्त्रग्नं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्वं त्व',
  'task_type': 'en2sa'},
 {'instruction': 'Translate the following English text into Sanskrit.',
  'input': 'And presenting him with many a charming fountain, trees will delight Rāma at the tops of mountains.* Where Rama is, there is nor fear or failure. That mighty-armed son of Dasaratha is heroic. Let 

## 7. Evaluation

Computes BLEU and chrF++ over the prediction files, broken down by
`task_type`, plus a before-vs-after comparison table and a simple heuristic
error-flagging pass for qualitative failure analysis.

chrF++ is reported for all task types since it's more forgiving of
Sanskrit's rich morphology/compounding than word-level BLEU; for
non-translation task types (explain/qa/summary) read these numbers as a
rough proxy only — n-gram metrics are a weak signal for open-ended
generation (see REPORT.md section 6).


In [25]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def compute_metrics(rows):
    by_type = defaultdict(list)
    for r in rows:
        by_type[r.get("task_type", "unknown")].append(r)

    metrics = {}
    for task_type, items in by_type.items():
        hyps = [it["prediction"] for it in items]
        refs = [it["reference"] for it in items]
        bleu = sacrebleu.corpus_bleu(hyps, [refs])
        chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2)  # chrF++
        metrics[task_type] = {
            "n": len(items),
            "bleu": round(bleu.score, 2),
            "chrf++": round(chrf.score, 2),
        }
    return metrics


def flag_heuristic_errors(rows):
    """Cheap heuristics to surface candidate failure cases for manual review:
    empty predictions, verbatim-echo of input, much-longer-than-reference
    (rambling/hallucination), and repeated n-grams (degenerate repetition)."""
    flagged = []
    for r in rows:
        pred = r["prediction"].strip()
        inp = r.get("input", "").strip()
        ref = r["reference"].strip()
        reasons = []

        if len(pred) < 2:
            reasons.append("empty_or_near_empty")
        if inp and pred == inp:
            reasons.append("echoed_input_verbatim")
        if ref and len(pred) > 3 * max(len(ref), 1):
            reasons.append("much_longer_than_reference")
        words = pred.split()
        if len(words) >= 6:
            trigrams = [tuple(words[i:i + 3]) for i in range(len(words) - 2)]
            if len(trigrams) > 0 and len(set(trigrams)) < len(trigrams) * 0.6:
                reasons.append("repetitive_degeneration")

        if reasons:
            flagged.append({**r, "flags": reasons})
    return flagged


In [26]:
def build_eval_report(base_predictions, finetuned_predictions, out_path, n_examples=8):
    ft_rows = load_jsonl(finetuned_predictions)
    ft_metrics = compute_metrics(ft_rows)
    ft_flagged = flag_heuristic_errors(ft_rows)

    base_rows, base_metrics = None, None
    if base_predictions:
        base_rows = load_jsonl(base_predictions)
        base_metrics = compute_metrics(base_rows)

    lines = ["# Evaluation Report\n"]

    lines.append("## Quantitative Metrics (fine-tuned model)\n")
    lines.append("| Task Type | N | BLEU | chrF++ |")
    lines.append("|---|---|---|---|")
    for t, m in sorted(ft_metrics.items()):
        lines.append(f"| {t} | {m['n']} | {m['bleu']} | {m['chrf++']} |")
    lines.append("")

    if base_metrics:
        lines.append("## Before vs After (base model vs fine-tuned)\n")
        lines.append("| Task Type | BLEU (base) | BLEU (fine-tuned) | Δ BLEU | "
                      "chrF++ (base) | chrF++ (fine-tuned) | Δ chrF++ |")
        lines.append("|---|---|---|---|---|---|---|")
        for t in sorted(ft_metrics.keys()):
            bm = base_metrics.get(t, {"bleu": float("nan"), "chrf++": float("nan")})
            fm = ft_metrics[t]
            d_bleu = fm["bleu"] - bm["bleu"] if bm["bleu"] == bm["bleu"] else float("nan")
            d_chrf = fm["chrf++"] - bm["chrf++"] if bm["chrf++"] == bm["chrf++"] else float("nan")
            lines.append(
                f"| {t} | {bm['bleu']} | {fm['bleu']} | {d_bleu:+.2f} | "
                f"{bm['chrf++']} | {fm['chrf++']} | {d_chrf:+.2f} |"
            )
        lines.append("")

    lines.append(f"## Heuristically Flagged Failure Cases ({len(ft_flagged)} / {len(ft_rows)})\n")
    lines.append("These are candidates for manual review, not confirmed errors.\n")
    for f in ft_flagged[:15]:
        lines.append(f"- **Flags:** {', '.join(f['flags'])} | **Task:** {f['task_type']}")
        lines.append(f"  - Input: `{f['input'][:120]}`")
        lines.append(f"  - Reference: `{f['reference'][:120]}`")
        lines.append(f"  - Prediction: `{f['prediction'][:120]}`")
    lines.append("")

    lines.append(f"## Qualitative Before/After Examples (first {n_examples})\n")
    for i in range(min(n_examples, len(ft_rows))):
        r = ft_rows[i]
        lines.append(f"**Example {i+1}** ({r['task_type']})")
        lines.append(f"- Instruction: {r['instruction']}")
        lines.append(f"- Input: `{r['input']}`")
        lines.append(f"- Reference: `{r['reference']}`")
        if base_rows and i < len(base_rows):
            lines.append(f"- Base model output: `{base_rows[i]['prediction']}`")
        lines.append(f"- Fine-tuned model output: `{r['prediction']}`")
        lines.append("")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Wrote report to {out_path}")
    print(json.dumps(ft_metrics, indent=2))
    return ft_metrics


eval_metrics = build_eval_report(
    base_predictions="eval/predictions_base.jsonl",
    finetuned_predictions="eval/predictions.jsonl",
    out_path="eval/report.md",
    n_examples=8,
)


Wrote report to eval/report.md
{
  "en2sa": {
    "n": 45,
    "bleu": 0.07,
    "chrf++": 9.41
  },
  "explain": {
    "n": 15,
    "bleu": 6.13,
    "chrf++": 23.37
  },
  "qa": {
    "n": 10,
    "bleu": 0.96,
    "chrf++": 12.2
  },
  "sa2en": {
    "n": 24,
    "bleu": 0.6,
    "chrf++": 17.37
  },
  "summary": {
    "n": 6,
    "bleu": 1.33,
    "chrf++": 16.77
  }
}


## 8. Try it yourself

Quick ad-hoc single-prompt inference against the fine-tuned model, for
manual sanity-checking before writing up the report.


In [27]:
m, tok = load_inference_model(CONFIG["base_model"], CONFIG["output_dir"])
pred = generate(m, tok, "Translate the following Sanskrit text into English.", "यत्र नार्यस्तु पूज्यन्ते रमन्ते तत्र देवताः।")
print(pred)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

And there, O goddess, the women worship the deities there.


In [28]:
# यत्र नार्यस्तु पूज्यन्ते रमन्ते तत्र देवताः।",
#          "Where women are honored, there the divine beings rejoice."),

In [29]:
import sacrebleu

def score_and_rank(predictions_file, task_type_filter=None, top_n=5, bottom_n=5):
    """Score every prediction individually (sentence-level chrF++), rank them,
    and return the best- and worst-scoring examples. This is how you find
    genuine successes/failures instead of eyeballing and cherry-picking."""
    rows = load_jsonl(predictions_file)
    if task_type_filter:
        rows = [r for r in rows if r.get("task_type") == task_type_filter]

    for r in rows:
        # sentence-level chrF++ as a per-example quality proxy
        r["_chrf"] = sacrebleu.sentence_chrf(r["prediction"], [r["reference"]], word_order=2).score

    ranked = sorted(rows, key=lambda r: r["_chrf"], reverse=True)

    print(f"=== TOP {top_n} (highest chrF++) — candidates for 'successful cases' in report ===\n")
    for r in ranked[:top_n]:
        print(f"chrF++={r['_chrf']:.1f}  task={r['task_type']}")
        print(f"  Input:      {r['input']}")
        print(f"  Reference:  {r['reference']}")
        print(f"  Prediction: {r['prediction']}\n")

    print(f"=== BOTTOM {bottom_n} (lowest chrF++) — candidates for 'failure analysis' ===\n")
    for r in ranked[-bottom_n:]:
        print(f"chrF++={r['_chrf']:.1f}  task={r['task_type']}")
        print(f"  Input:      {r['input']}")
        print(f"  Reference:  {r['reference']}")
        print(f"  Prediction: {r['prediction']}\n")

    return ranked


# Run across all task types together, and per task type if you want cleaner examples per category
all_ranked = score_and_rank("eval/predictions.jsonl", top_n=5, bottom_n=5)

# Optional: per-task-type breakdown, e.g. just translation direction
# sa2en_ranked = score_and_rank("eval/predictions.jsonl", task_type_filter="sa2en", top_n=3, bottom_n=3)

=== TOP 5 (highest chrF++) — candidates for 'successful cases' in report ===

chrF++=45.5  task=explain
  Input:      ततस्तत्र प्रविष्टस्य कौसल्याया निवेशनम्। अधिरुह्यापि शयनं बभूव लुलितं मनः॥
  Reference:  This verse conveys: Having entered Kausalyā's apartment, the king having laid himself on the bed, was overwhelmed with emotion.
  Prediction: This verse conveys: And thereupon, the king, having entered the abode of Kausalyā, was struck with grief, and was overcome with sorrow.

chrF++=37.4  task=explain
  Input:      तस्यैषा लोकनाथस्य धर्मपत्नी यशस्विनी॥ सीता नाम वरारोहा यां त्वं हर्तुमिच्छसि।
  Reference:  This verse conveys: This exquisitely beautiful and far-famed Sītā whom you are about to steal away, is the married wife of that lord of men.
  Prediction: This verse conveys: This is the virtuous wife of the Lord of the universe, Sītā, and you should come to me.

chrF++=34.0  task=qa
  Input:      देवैस्तदा समागम्य सर्षिसङ्घः सचारणैः॥ याचितौ प्रशमं तत्र जग्मतुस्तौ सुरोत्तमौ।
  Re

## 8. Save / Export

The LoRA adapter (small, a few hundred MB at most) is saved under
`outputs/lora-sanskrit/`. Zip and download it, or push it to the Hugging Face
Hub, to include it in your submission per the assignment's deliverables
checklist.


In [32]:
import shutil
shutil.make_archive("lora-sanskrit-adapter", "zip", "outputs/lora-sanskrit")
print("Zipped adapter -> lora-sanskrit-adapter.zip")

from google.colab import files
files.download("lora-sanskrit-adapter.zip")


Zipped adapter -> lora-sanskrit-adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# Optional: push adapter to the Hugging Face Hub instead of downloading a zip
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo("your-username/sanskrit-llama-lora", exist_ok=True)
# api.upload_folder(folder_path="outputs/lora-sanskrit", repo_id="your-username/sanskrit-llama-lora")



## Next Steps / What We'd Improve With More Time

See `REPORT.md` section 9 for the full list. Highlights:
- **Add a repetition penalty / no-repeat-ngram constraint at generation
  time** (or drop pure greedy decoding for longer outputs) — 41 of the 100
  held-out predictions degenerate into repeating the same word or phrase
  until they hit `max_new_tokens`. This is the single cheapest, highest-
  leverage fix available and doesn't require retraining anything.
- Tokenizer adaptation (extend vocab with Devanagari-aware merges) given the
  fragmentation ratio measured in section 4 above.
- Larger, more diverse Sanskrit sources (GRETIL, Digital Corpus of Sanskrit,
  AI4Bharat) beyond itihasa's epic-poetry register, to generalize past
  Ramayana/Mahabharata style text.
- Human evaluation (native Sanskrit speaker) rather than only automatic
  metrics, especially for the explanation/QA tasks.
- LoRA-vs-full-fine-tune and quantization ablations (bonus items).
